# SLAYER-OCR: RF-DETR layout pilot v1

Prywatny trening na 60 zweryfikowanych stronach. Ustaw **Runtime -> Change runtime type -> T4 GPU**, wybierz **Run all** i wgraj dokładnie plik `slayer-layout-rfdetr-private-v1-20260929.zip`. Notebook nie publikuje danych ani modelu. Split `valid` jest zbiorem deweloperskim, nie końcowym testem PolOCRBench.

In [ ]:
%pip install -q --no-cache-dir "rfdetr[train]==1.11.0"

In [ ]:
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
from pathlib import Path, PurePosixPath
import shutil
import stat
import sys
import zipfile

import torch

RFDETR_VERSION = '1.11.0'
DATASET_BUILDER_REVISION = 'b6570a7829fc96651c7a41db8cae35e59c888460'
EXPECTED_DATASET_SHA256 = 'f0a157c79756276e9a82e4f8010521190bf72dc8b459c81c3806e840681a0c33'
EXPECTED_CANDIDATE_COCO_SHA256 = '31490eb8b64668f02611b90879ad1221bc0c74ef46136f363e71cba60f069451'
EXPECTED_REVIEW_SHA256 = 'fe3904bc856edfa669bbc4b983506cb5b86658ea7a26c12001b980eceff8b448'

assert importlib.metadata.version('rfdetr') == RFDETR_VERSION
assert torch.cuda.is_available(), 'Wybierz runtime GPU i uruchom notebook od początku.'
print('ENVIRONMENT_OK', torch.cuda.get_device_name(0), 'rfdetr', RFDETR_VERSION)

In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, 'Wgraj dokładnie jeden prywatny ZIP RF-DETR.'
upload_name, upload_bytes = next(iter(uploaded.items()))
assert upload_name.endswith('.zip'), 'Oczekiwano pliku ZIP.'
actual_dataset_sha256 = hashlib.sha256(upload_bytes).hexdigest()
assert actual_dataset_sha256 == EXPECTED_DATASET_SHA256, (
    'Hash ZIP-a nie pasuje do zweryfikowanego pakietu', actual_dataset_sha256)
archive_path = Path('/content') / upload_name
archive_path.write_bytes(upload_bytes)
del upload_bytes, uploaded
print('PRIVATE_DATASET_ARCHIVE_OK', actual_dataset_sha256)

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

extract_root = Path('/content/slayer-layout-rfdetr-private-v1')
assert not extract_root.exists(), 'Katalog danych już istnieje; uruchom świeżą sesję.'
extract_root.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    infos = archive.infolist()
    names = [item.filename for item in infos]
    assert len(names) == len(set(names)), 'ZIP zawiera zduplikowane ścieżki.'
    assert sum(item.file_size for item in infos) < 500_000_000, 'ZIP jest zbyt duży.'
    for item in infos:
        path = PurePosixPath(item.filename)
        mode = (item.external_attr >> 16) & 0o170000
        assert not path.is_absolute() and '..' not in path.parts
        assert '\\' not in item.filename and ':' not in item.filename
        assert mode != stat.S_IFLNK, 'Symlink w ZIP-ie jest niedozwolony.'
    archive.extractall(extract_root)

provenance_paths = list(extract_root.rglob('provenance.json'))
assert len(provenance_paths) == 1, 'Nie znaleziono jednoznacznego provenance.json.'
dataset_root = provenance_paths[0].parent
provenance = json.loads(provenance_paths[0].read_text(encoding='utf-8'))
assert provenance['schema'] == 'slayer-layout-rfdetr-dataset-v1'
assert provenance['release_status'] == 'private-training-candidate-not-published'
assert provenance['source']['candidate_coco_sha256'] == EXPECTED_CANDIDATE_COCO_SHA256
assert provenance['source']['review_input_sha256'] == EXPECTED_REVIEW_SHA256
assert provenance['train']['pages'] == 48 and provenance['valid']['pages'] == 12
assert provenance['collection_overlap'] == [] and provenance['image_hash_overlap'] == []

expected_files = {}
for line in (dataset_root / 'checksums.sha256').read_text(encoding='utf-8').splitlines():
    digest, relative = line.split('  ', 1)
    assert relative not in expected_files
    path = (dataset_root / relative).resolve()
    assert path.is_relative_to(dataset_root.resolve()) and path.is_file()
    assert sha256_file(path) == digest, relative
    expected_files[relative] = digest
actual_files = {
    path.relative_to(dataset_root).as_posix()
    for path in dataset_root.rglob('*')
    if path.is_file() and path.name != 'checksums.sha256'
}
assert actual_files == set(expected_files), 'Nieoczekiwane lub brakujące pliki danych.'
for split in ('train', 'valid'):
    annotations = json.loads(
        (dataset_root / split / '_annotations.coco.json').read_text(encoding='utf-8'))
    assert annotations['info']['schema'] == 'slayer-layout-rfdetr-dataset-v1'
    assert all((dataset_root / split / row['file_name']).is_file()
               for row in annotations['images'])
print('PRIVATE_DATASET_VERIFIED', json.dumps({
    'train': provenance['train'], 'valid': provenance['valid']
}, ensure_ascii=False, indent=2))

In [ ]:
from rfdetr import RFDETRSmall

RUN_CONFIG = {
    'model': 'RFDETRSmall',
    'epochs': 100,
    'batch_size': 4,
    'grad_accum_steps': 4,
    'eval_batch_size': 4,
    'resolution': 512,
    'lr': 1e-4,
    'lr_encoder': 1.5e-4,
    'early_stopping': True,
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 0.001,
    'skip_best_epochs': 3,
    'checkpoint_interval': 10,
    'seed': 20260929,
}
output_dir = Path('/content/slayer-rfdetr-layout-v1-output')
assert not output_dir.exists(), 'Katalog treningu już istnieje; uruchom świeżą sesję.'
model = RFDETRSmall()
model.train(
    dataset_dir=str(dataset_root),
    output_dir=str(output_dir),
    device='cuda',
    tensorboard=False,
    wandb=False,
    log_per_class_metrics=True,
    **RUN_CONFIG,
)
print('TRAINING_FINISHED', output_dir)

In [ ]:
def jsonable(value):
    if isinstance(value, dict):
        return {str(key): jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    if hasattr(value, 'item'):
        return value.item()
    return value

final_metrics = jsonable(model.evaluate(
    dataset_dir=str(dataset_root), split='val', eval_batch_size=4,
    resolution=RUN_CONFIG['resolution'], device='cuda',
    log_per_class_metrics=True, tensorboard=False, wandb=False,
))
run_manifest = {
    'schema': 'slayer-layout-rfdetr-run-v1',
    'status': 'completed',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'dataset_zip_sha256': EXPECTED_DATASET_SHA256,
    'dataset_builder_revision': DATASET_BUILDER_REVISION,
    'rfdetr_version': importlib.metadata.version('rfdetr'),
    'torch_version': torch.__version__,
    'python_version': sys.version,
    'gpu': torch.cuda.get_device_name(0),
    'config': RUN_CONFIG,
    'metrics': final_metrics,
    'claim_boundary': 'Private development pilot; not final-test or SOTA evidence.',
}
(output_dir / 'final_metrics.json').write_text(
    json.dumps(final_metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(output_dir / 'run_manifest.json').write_text(
    json.dumps(run_manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert (output_dir / 'checkpoint_best_total.pth').is_file(), 'Brak końcowego checkpointu.'
print('FINAL_METRICS', json.dumps(final_metrics, ensure_ascii=False, indent=2))

In [ ]:
package_root = Path('/content/slayer-rfdetr-layout-v1-packages')
assert not package_root.exists()
evidence_dir = package_root / 'evidence'
model_dir = package_root / 'model'
evidence_dir.mkdir(parents=True)
model_dir.mkdir(parents=True)

for source, name in [
    (output_dir / 'run_manifest.json', 'run_manifest.json'),
    (output_dir / 'final_metrics.json', 'final_metrics.json'),
    (dataset_root / 'provenance.json', 'dataset_provenance.json'),
    (dataset_root / 'checksums.sha256', 'dataset_checksums.sha256'),
]:
    shutil.copyfile(source, evidence_dir / name)
training_config = output_dir / 'training_config.json'
if training_config.is_file():
    shutil.copyfile(training_config, evidence_dir / training_config.name)
for path in output_dir.rglob('*'):
    if path.is_file() and path.suffix.lower() in {'.json', '.csv'}:
        relative_name = '__'.join(path.relative_to(output_dir).parts)
        target = evidence_dir / relative_name
        if not target.exists():
            shutil.copyfile(path, target)

for name in ['checkpoint_best_total.pth', 'training_config.json',
             'run_manifest.json', 'final_metrics.json']:
    source = output_dir / name
    if source.is_file():
        shutil.copyfile(source, model_dir / name)
for directory in (evidence_dir, model_dir):
    paths = sorted(path for path in directory.rglob('*') if path.is_file())
    (directory / 'checksums.sha256').write_text(''.join(
        f'{sha256_file(path)}  {path.relative_to(directory).as_posix()}\n'
        for path in paths), encoding='utf-8')

evidence_zip = Path(shutil.make_archive(
    '/content/slayer-rfdetr-layout-v1-evidence', 'zip', evidence_dir))
model_zip = Path(shutil.make_archive(
    '/content/slayer-rfdetr-layout-v1-model', 'zip', model_dir))
print('EVIDENCE_READY', evidence_zip, sha256_file(evidence_zip))
print('MODEL_READY', model_zip, sha256_file(model_zip))

## Pobranie wyników

Uruchom kolejno dwie komórki. Zachowaj oba ZIP-y bez zmiany nazw.

In [ ]:
files.download(str(evidence_zip))

In [ ]:
files.download(str(model_zip))